# 低内存开销的 Dropout

本教程将编写一个节省内存的 Dropout 实现，其状态仅由一个 int32 随机种子组成。传统实现通常需要保存一个与输入形状相同的掩码张量。

你将学习：

- 使用 PyTorch 朴素实现 Dropout 的局限。
- Triton 中的并行伪随机数生成。

---

来源：[Triton 官方教程](https://triton-lang.org/main/getting-started/tutorials/04-low-memory-dropout.html)，版本为 2026-10-16 下载的官网 main 文档。

中文练习版：保留说明与公式；原有代码单元及说明中的独立代码块均保留为空白代码框；移除代码、运行输出、测量结果与图表。空白框供自行填写，未运行教程。原文许可见 [TRITON-LICENSE.txt](TRITON-LICENSE.txt)。

## 基线实现

Dropout 最初由参考文献 [SRIVASTAVA2014] 提出，作为一种正则化方法，改善深度神经网络在数据较少时的表现。

它接受一个向量，并输出同形状的向量。每个元素以概率 $p$ 被置零，否则保留输入值。这使网络在只有部分输入元素可用时也要保持良好表现，平均保留比例为 $1-p$。

评估时将 $p$ 设为零，使用完整网络。如果不调整尺度，训练和评估时输出的幅度可能发生变化，例如影响后续 Softmax 的有效温度。因此，训练时通常将保留的输出乘以 $1/(1-p)$，使输出的期望尺度保持一致。

译注：原文以“范数保持一致”描述这一缩放；更准确地说，这种缩放保持的是逐元素期望，并不保证每次采样的范数相同。

首先来看基线实现。

## 基于随机种子的 Dropout

上面的实现可以正常工作，但有两点不便：反向传播需要保存 Dropout 掩码；使用重计算或检查点机制时，随机状态的管理会变得复杂。相关背景可参考 [PyTorch 检查点文档](https://pytorch.org/docs/stable/checkpoint.html) 中对 `preserve_rng_state` 的说明。

本教程介绍另一种实现，以降低内存占用、减少数据搬运，并简化多次内核调用之间的随机性管理。

Triton 提供了方便的伪随机数生成接口。本教程使用 `triton.language.rand`：给定一个种子和一块 int32 偏移量，它会生成一块在 [0, 1) 区间均匀分布的 float32 数值。Triton 还支持其他随机数生成策略。

> 注：Triton 的伪随机数生成器基于 Philox 算法，参见 [SALMON2011]。

下面将这些部分组合起来。

在相同的种子和元素偏移映射下，内核会生成相同的 Dropout 掩码。若想进一步了解 GPU 编程中的伪随机数应用，可以阅读 Triton 自身的 `python/triton/language/random.py`。

## 练习

1. 将内核扩展到矩阵输入，并使用一个种子向量，为每一行指定独立的种子。
2. 添加对非连续步长的支持。
3. 挑战：实现稀疏 Johnson–Lindenstrauss 变换内核，每次通过随机种子即时生成投影矩阵。

## 参考文献

- [SALMON2011] John K. Salmon、Mark A. Moraes、Ron O. Dror、David E. Shaw，*Parallel Random Numbers: As Easy as 1, 2, 3*（并行随机数：像 1、2、3 一样简单），2011。
- [SRIVASTAVA2014] Nitish Srivastava、Geoffrey Hinton、Alex Krizhevsky、Ilya Sutskever、Ruslan Salakhutdinov，*Dropout: A Simple Way to Prevent Neural Networks from Overfitting*（Dropout：防止神经网络过拟合的一种简单方法），JMLR，2014。